In [ ]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import SGDClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# -------------------------
# 1. Load Dataset
# -------------------------

df = pd.read_csv("model_training_dataset.csv")

print("Dataset Shape:", df.shape)

# -------------------------
# 2. Fix Target Classes
# -------------------------

# IMMEDIATE has very few samples,
# merge it into HIGH for stable training
df["response_priority"] = df["response_priority"].replace(
    "IMMEDIATE",
    "HIGH"
)

print("\nTarget Distribution:")
print(df["response_priority"].value_counts())

# -------------------------
# 3. Select Features
# -------------------------

features = [
    "final_risk_score",
    "Population",
    "affected_population",
    "total_health_facilities",
    "total_schools"
]

X = df[features]

y = df["response_priority"]

# -------------------------
# 4. Train-Test Split
# -------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining rows:", len(X_train))
print("Testing rows:", len(X_test))

# -------------------------
# 5. Build Pipeline
# -------------------------

model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "classifier",
        SGDClassifier(
            loss="log_loss",
            class_weight="balanced",
            max_iter=2000,
            tol=1e-3,
            random_state=42
        )
    )
])

# -------------------------
# 6. Train
# -------------------------

model.fit(X_train, y_train)

# -------------------------
# 7. Test
# -------------------------

predictions = model.predict(X_test)

accuracy = accuracy_score(
    y_test,
    predictions
)

print("\nAccuracy:", round(accuracy * 100, 2), "%")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        predictions,
        zero_division=0
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test,
        predictions
    )
)

# -------------------------
# 8. Save Model
# -------------------------

joblib.dump(
    model,
    "disaster_priority_model.pkl"
)

print("\nModel saved:")
print("disaster_priority_model.pkl")

: 